In [ ]:
import sys

sys.path.insert(1, "/home/xilinx/qick-qoc/board/")

from programs.qdac import *
from programs.randomized_benchmarking2_with_single_shot_readout import *
from programs.res_spec import *

from scipy.optimize import curve_fit

from tqdm.notebook import tqdm
import numpy as np
import matplotlib.pyplot as plt

%matplotlib notebook

## LOAD FIRMWARE

In [ ]:
# Load bitstream with custom overlay
soccfg = QickSoc(bitfile="/home/xilinx/jupyter_notebooks/qick/qick_lib/qick/qick_size_mod_2024MAY08.bit", external_clk=True)

## PHASE CONFIGURATION

In [ ]:
# Resonator
READOUT_RESONATOR_GAIN = 4500   # Power to resonator
READOUT_RESONATOR_FREQ = 311.03 # Resonator frequency [MHz]
READOUT_PULSE_LENGTH   = 10.0  # us2cycles value (needs calibration)

PHASE_CONFIG_NUM_AVERAGES = 1000  # number of averages
PHASE_CONFIG_RELAX_DELAY = 10

hw_cfg = {"res_ch": 1, "qubit_ch": 0, "storage_ch": 0}  # No JPA channel
readout_cfg = {
    "readout_length": soccfg.us2cycles(READOUT_PULSE_LENGTH),  # [Clock ticks]
    # "f_res": 99.775 + 0.18,  # [MHz]
    "res_phase": 0,
    "adc_trig_offset": 275,  # [Clock ticks]
    "frequency" : READOUT_RESONATOR_FREQ,
    "res_gain" : READOUT_RESONATOR_GAIN,
}

expt_cfg = {
    "reps": PHASE_CONFIG_NUM_AVERAGES,
    "relax_delay": PHASE_CONFIG_RELAX_DELAY,
    "start": 0,
    "step": 0,
    "expts": 1}
config = {**hw_cfg, **readout_cfg, **expt_cfg}  # combine configs

In [ ]:
phase_pts = np.linspace(0,360,361)

avg_i0 = np.empty([phase_pts.shape[0]])
avg_q0 = np.empty([phase_pts.shape[0]])
result = np.empty([phase_pts.shape[0]])

for p_idx, p in enumerate(tqdm(phase_pts)):
    config["res_phase"] = p
    rspec = SingleToneSpectroscopyProgram(soccfg, config)
    avgi, avgq = rspec.acquire(soccfg, load_pulses=True)
    avg_i0[p_idx] = avgi[0][0]
    avg_q0[p_idx] = avgq[0][0]

In [ ]:
# THE LAMEST VIDEO GAME EVER, MATCH THE ORANGE AND BLUE LINES TO WHERE THE SINUSOIDS CROSS (IS SPECIFIC TO THE SYSTEM...)
plt.figure()
plt.plot(phase_pts, avg_i0)
plt.plot(phase_pts, avg_q0)
plt.axhline(0)
plt.axhline(0.67,color='b')
plt.axhline(0,color='r')
plt.axhline(-0.67,color='orange')

In [ ]:
RES_PHASE = 315 # degrees

## FluxTuning

In [ ]:
RELAX_DELAY = 1000

# Qubit
QUBIT_FREQ        = 224.704e6  # Qubit frequency [Hz]
QUBIT_PERIOD      = 1./QUBIT_FREQ

# QICK
DAC_OUTCLOCK_RATE   = 6881.280e6
DAC_OUTCLOCK_PERIOD = 1/DAC_OUTCLOCK_RATE  # RFSoC DAC Sampling Interval from QICK config
ADC_OFFSET          = 275                 # Time-of-flight calibration

In [ ]:
# FLUX TUNING
FT_QDAC_CHANNEL   = 3
FT_NUM_AVERAGES   = 4000
FT_QUBIT_GAIN     = 1000      # Power to qubit
FT_PROBE_LENGTH   = 1e-6   # s
FT_FREQ           = 224.704e6 # Hz
#
FT_FLUX_TESTRANGE     =  0.08
FT_FLUX_TESTNUMPOINTS =  26

In [ ]:
def init_flux_test_pulse(FT_FREQ):
    fluxtestpulsets = np.arange(0., FT_PROBE_LENGTH, DAC_OUTCLOCK_PERIOD)
    fluxtestpulse   = Xd2Gen.getGaussianSinusoidPulse(fluxtestpulsets, FT_FREQ, FT_PROBE_LENGTH/4)
    return fluxtestpulse

def run_flux_test(prev_flux): 
    results = []
    fluxes  = np.linspace(prev_flux-FT_FLUX_TESTRANGE/2, prev_flux+FT_FLUX_TESTRANGE/2, FT_FLUX_TESTNUMPOINTS)
    for flux in tqdm(fluxes, leave=False):
        qdac.set_voltage(FT_QDAC_CHANNEL,flux,dwell=0.1)
        #
        cfg    = genFluxTuningConfigs(FLUX_TEST_PULSE)
        rbprog = RandomizedBenchmarking2OriginalProgram(soccfg, cfg)
        avgi, avgq = rbprog.acquire(soccfg, load_pulses=True)
        #
        results.append(avgi[0][0])
    return fluxes, results

def gaussian(fluxes, b, c, amp, offset):
    f = amp*np.exp(-(fluxes-b)**2/(2*c**2))+offset
    return f

def get_optimal_flux_fit(fluxes, results):
    popt, pcov = curve_fit(gaussian, fluxes, results, maxfev=10000)
    return popt

def run_flux_calibration(currentoptimalflux):
    fluxes, results = run_flux_test(currentoptimalflux)
    popt = get_optimal_flux_fit(fluxes, results)
    new_optimal_flux = np.round(popt[0], 5)
    qdac.set_voltage(FT_QDAC_CHANNEL,new_optimal_flux,dwell=0.1)
    return fluxes, results, new_optimal_flux

In [ ]:
def genFluxTuningConfigs(arbitrarypulse):
    qubit_cfg = {
        "qubit_ch"         : 0, # 4 in QICK supplied build
        "qubit_pulse_name" : "arbitrary_pulse",
        "qubit_i_data"     : (2**15-2)*arbitrarypulse,
        "qubit_q_data"     : np.zeros_like(arbitrarypulse),
        "qubit_gain"       : FT_QUBIT_GAIN,
    }
    readout_cfg = {
        "res_ch"             : 1, # 6 in QICK supplied build
        "res_freq"           : READOUT_RESONATOR_FREQ,  # Resonator frequency [MHz]
        "res_phase"          : RES_PHASE,
        "res_gain"           : READOUT_RESONATOR_GAIN,  # Power to resonator
        "res_readout_length" : soccfg.us2cycles(READOUT_PULSE_LENGTH),  # [Clock ticks]
        "adc_trig_offset"    : ADC_OFFSET,  # [Clock ticks]
    }
    expt_cfg = {
        "reps"        : FT_NUM_AVERAGES,
        "rounds"      : 1,
        "relax_delay" : RELAX_DELAY,
    }
    cfg = {**readout_cfg, **qubit_cfg, **expt_cfg}  # combine configs
    return cfg

In [ ]:
qdac = QDAC_II()

In [ ]:
FLUX_TEST_PULSE = init_flux_test_pulse(FT_FREQ)
fluxes, flux_test_results, currentoptimalflux = run_flux_calibration(currentoptimalflux)
print(currentoptimalflux)

In [ ]:
qdac.close()

In [ ]:
plt.figure()
plt.plot(fluxes, flux_test_results)

## PART 3 AMPLITUDE RABI SINGLE SHOT

In [ ]:
RELAX_DELAY = 1000

# Qubit
QUBIT_FREQ        = 224.704e6  # Qubit frequency [Hz]
QUBIT_PERIOD      = 1./QUBIT_FREQ

# QICK
DAC_OUTCLOCK_RATE   = 6881.280e6
DAC_OUTCLOCK_PERIOD = 1/DAC_OUTCLOCK_RATE  # RFSoC DAC Sampling Interval from QICK config
ADC_OFFSET          = 275                 # Time-of-flight calibration

In [ ]:
PULSE_SAMPLERATE = 100*DAC_OUTCLOCK_RATE
PULSE_NAMES      = ["n200"]
PULSES           = [Xd2Gen.getXd2Pulse("/home/xilinx/data/rb_2024OCT/pronto_{}_2024OCT14.csv".format(name)) for name in PULSE_NAMES]
PULSE_DURATIONS  = [len(pulse)/PULSE_SAMPLERATE for pulse in PULSES]

In [ ]:
AMP_LOW  = 0  # gain
AMP_HIGH = 2000  # gain
AMP_STEP = 10  # d_gain

NUM_AVERAGES = 1000

In [ ]:
def genRandomizedBenchmarkingConfigs(arbitrarypulse):
    qubit_cfg = {
        "qubit_ch"         : 0, # 4 in QICK supplied build
        "qubit_pulse_name" : "arbitrary_pulse",
        "qubit_i_data"     : (2**15-2)*arbitrarypulse,
        "qubit_q_data"     : np.zeros_like(arbitrarypulse),
    }
    readout_cfg = {
        "res_ch"             : 1, # 6 in QICK supplied build
        "res_freq"           : READOUT_RESONATOR_FREQ,  # Resonator frequency [MHz]
        "res_phase"          : RES_PHASE,
        "res_gain"           : READOUT_RESONATOR_GAIN,  # Power to resonator
        "res_readout_length" : soccfg.us2cycles(READOUT_PULSE_LENGTH),  # [Clock ticks]
        "adc_trig_offset"    : ADC_OFFSET,  # [Clock ticks]
        "delta_t"            : 0,
    }
    expt_cfg = {
        "start": AMP_LOW,
        "step": AMP_STEP,
        "expts": int((AMP_HIGH - AMP_LOW) / AMP_STEP) + 1,
        "reps"            : NUM_AVERAGES,
        "relax_delay"     : RELAX_DELAY,
    }
    cfg = {**readout_cfg, **qubit_cfg, **expt_cfg}  # combine configs
    return cfg

In [ ]:
xd2gate                          = ['X/2','X/2']
PulseGen.genArbitraryPulseStart(PULSES[0], PULSE_DURATIONS[0], QUBIT_PERIOD, DAC_OUTCLOCK_PERIOD)
PulseGen.genArbitraryPulseAddGates(xd2gate)
arbitrarypulsets, arbitrarypulse = PulseGen.genArbitraryPulseGetOutput()
cfg                              = genRandomizedBenchmarkingConfigs(arbitrarypulse)
rbprog                           = RandomizedBenchmarking2GainSweepSingleShot(soccfg, cfg)
xpts, shots_i0, shots_q0         = rbprog.acquire(soccfg, load_pulses=True, progress=True)

In [ ]:
THRESHOLD = 0.9
success_runsum  = np.zeros(len(xpts))
success_numelem = np.zeros(len(xpts))
for ii in range(len(xpts)):
    for jj in range(NUM_AVERAGES):
        if(shots_i0[ii][jj][0] < THRESHOLD): # ONLY CARE IF DATA WAS IN THE GROUND STATE TO BEGIN WITH
            success_numelem[ii] += 1
            if(shots_i0[ii][jj][1] < THRESHOLD):
                success_runsum[ii] += 1
            else:
                success_runsum[ii] += 0
success_prob = [success_runsum[ii]/success_numelem[ii] for ii in range(len(xpts))]

In [ ]:
plt.figure()
plt.plot(xpts, success_prob)
plt.ylim((0,1))

## PART 2 ZIG ZAG

In [ ]:
RELAX_DELAY = 1000

# Qubit
QUBIT_FREQ        = 224.704e6  # Qubit frequency [Hz]
QUBIT_PERIOD      = 1./QUBIT_FREQ

# QICK
DAC_OUTCLOCK_RATE   = 6881.280e6
DAC_OUTCLOCK_PERIOD = 1/DAC_OUTCLOCK_RATE  # RFSoC DAC Sampling Interval from QICK config
ADC_OFFSET          = 275                 # Time-of-flight calibration

In [ ]:
PULSE_SAMPLERATE = 100*DAC_OUTCLOCK_RATE
PULSE_NAMES      = ["n200"]
PULSES           = [Xd2Gen.getXd2Pulse("/home/xilinx/data/rb_2024OCT/pronto_{}_2024OCT14.csv".format(name)) for name in PULSE_NAMES]
PULSE_DURATIONS  = [len(pulse)/PULSE_SAMPLERATE for pulse in PULSES]

In [ ]:
PULSE_PId2_GAIN_GUESSES = [400]

GAIN_PERCENTZOOM = 0.5
NUM_AVERAGES = 2000

NUM_LOW      = 1
NUM_HIGH     = 81
NUM_STEPSIZE = 2
pulse_nums   = np.arange(NUM_LOW, NUM_HIGH, NUM_STEPSIZE)

In [ ]:
# Checking that everything is OK
assert NUM_LOW % 2 == 1, "NUM_LOW should be odd!"
assert NUM_HIGH % 2 == 1, "NUM_HIGH should be odd!"
assert NUM_STEPSIZE % 2 == 0, "NUM_STEP should be even!"

In [ ]:
def genConfigs(arbitrarypulse, expectedgain):
    minval   = max(0,       expectedgain*(1-GAIN_PERCENTZOOM))
    maxval   = min(2**15-1, expectedgain*(1+GAIN_PERCENTZOOM))
    gainstep = max(1,       int(expectedgain*GAIN_PERCENTZOOM/35))
    gains    = np.arange(minval,maxval,gainstep,dtype=int)
    qubit_cfg = {
        "qubit_ch"         : 0, # 4 in QICK supplied build
        "qubit_pulse_name" : "arbitrary_pulse",
        "qubit_i_data"     : (2**15-2)*arbitrarypulse,
        "qubit_q_data"     : np.zeros_like(arbitrarypulse),
        "qubit_gain"       : gains[0] #expected_gain - int(GAIN_NUM_POINTS*GAIN_STEP/2),
    }
    readout_cfg = {
        "res_ch"             : 1, # 6 in QICK supplied build
        "res_freq"           : READOUT_RESONATOR_FREQ,  # Resonator frequency [MHz]
        "res_phase"          : RES_PHASE,
        "res_gain"           : READOUT_RESONATOR_GAIN,  # Power to resonator
        "res_readout_length" : soccfg.us2cycles(READOUT_PULSE_LENGTH),  # [Clock ticks]
        "adc_trig_offset"    : ADC_OFFSET,  # [Clock ticks]
    }
    expt_cfg = {
        "start": gains[0],
        "step":  gainstep,
        "expts":  len(gains),
        "reps"            : NUM_AVERAGES,
        "relax_delay"     : RELAX_DELAY,
        "delta_t" : 0
    }
    cfg = {**readout_cfg, **qubit_cfg, **expt_cfg}  # combine configs
    return cfg

In [ ]:
qdac = QDAC_II()
pulses_result_arrays = [None]*len(PULSE_DURATIONS)
for ii in range(len(PULSE_DURATIONS)):
    result_array = []
    for jj, num_xd2_gates in enumerate(tqdm(pulse_nums)):
        if(jj%3==0):
            fluxes, flux_test_results, currentoptimalflux = run_flux_calibration(currentoptimalflux)
            print(currentoptimalflux)
        #
        xd2gates = ['X/2']*num_xd2_gates
        #
        PulseGen.genArbitraryPulseStart(PULSES[ii], PULSE_DURATIONS[ii], QUBIT_PERIOD, DAC_OUTCLOCK_PERIOD)
        PulseGen.genArbitraryPulseAddGates(xd2gates)
        arbitrarypulsets, arbitrarypulse = PulseGen.genArbitraryPulseGetOutput()
        #
        fluxtestpulsets = np.arange(0., 10e-6, DAC_OUTCLOCK_PERIOD)
        warmup = Xd2Gen.getGaussianSinusoidPulse(fluxtestpulsets, 200e6, 10e-6)
        #
        warmup_plus_arbitrarypulse = np.concatenate((warmup,arbitrarypulse))
        #
        cfg        = genConfigs(arbitrarypulse, PULSE_PId2_GAIN_GUESSES[ii])
        rbprog     = RandomizedBenchmarking2GainSweepSingleShot(soccfg, cfg)
        result_array.append(rbprog.acquire(soccfg, load_pulses=True))
    pulses_result_arrays[ii] = result_array

In [ ]:
qdac.close()

In [ ]:
THRESHOLD = 0.9

single_shot_results = []
for nn in range(len(pulse_nums)):
    xpts, shots_i0, shots_q0 = pulses_result_arrays[0][nn]
    success_runsum  = np.zeros(len(xpts))
    success_numelem = np.zeros(len(xpts))
    for ii in range(len(xpts)):
        for jj in range(NUM_AVERAGES):
            if(shots_i0[ii][jj][0] < THRESHOLD): # ONLY CARE IF DATA WAS IN THE GROUND STATE TO BEGIN WITH
                success_numelem[ii] += 1
                if(shots_i0[ii][jj][1] < THRESHOLD):
                    success_runsum[ii] += 1
                else:
                    success_runsum[ii] += 0
    success_prob = [success_runsum[ii]/success_numelem[ii] for ii in range(len(xpts))]
    single_shot_results.append(success_prob)

In [ ]:
gt_gains  = xpts
heatmap   = np.asarray([success_prob for success_prob in single_shot_results])
std_array = [sum([(0.5-re)**2 for re in r]) for r in np.transpose(heatmap)]
argminv   = np.argmin(std_array)
print(gt_gains[argminv])

In [ ]:
plt.figure()
#plt.figure()
GAIN_PTS = xpts
heatmap   = np.asarray([success_prob for success_prob in single_shot_results])
plt.pcolormesh(
    np.linspace(min(GAIN_PTS), max(GAIN_PTS), len(GAIN_PTS)+1),
    np.linspace(NUM_LOW, NUM_HIGH, len(pulse_nums)+1),
    heatmap, cmap='PuBuGn',
    vmin=0.0,vmax=1.0,
)
plt.colorbar()
# Plot vertical/horizontal lines and show guess values
plt.axvline(x=gt_gains[argminv], c="r")
plt.xlabel('Pulse Amplitude')
plt.ylabel('Number of Pulses')
plt.title('pi/2 pulse calibration')

In [ ]:
currentoptimalflux